# فحص جودة ملف البيانات
شغّل الدفتر في Google Colab. اختر في خلية «مصدر البيانات» رفع ملفك أو مثال العملاء، وفعّل المرجع عند الرغبة في قياس الدقة. عند اكتمال تحميل الملف ستظهر قواعد أعمدته، ثم اضغط «تشغيل الفحص». يمكنك إعادة الفحص بعد تعديل القواعد وتنزيل لوحة HTML ونتائج JSON معًا.

ضع عناوين فريدة في الصف الأول. يدعم CSV وExcel ‏(.xlsx و.xlsm و.xls). يتم اختيار ورقة واحدة لكل ملف. تُقرأ خلايا الصيغ من آخر قيمة محفوظة في Excel. خزّن الهويات والأكواد كنص لحفظ الأصفار. تواريخ النص تقبل تنسيق ISO مثل 2026-11-18. تُستبعد الصفوف الفارغة بالكامل؛ أرقام السجلات في التقرير تبدأ من 1 بعد استبعادها. يُرفع الملف إلى جلسة Colab.

In [ ]:
import csv
import io
import json
import math
import re
from collections import Counter
from datetime import date, datetime
from pathlib import Path
from zoneinfo import ZoneInfo

DIMS = ["الاكتمال", "الصلاحية", "التفرّد", "الاتساق", "الدقة", "الحداثة"]

def normalize(value):
    if value is None:
        return ""
    if isinstance(value, datetime):
        return value.isoformat(sep=" ") if value.time() != datetime.min.time() else value.date().isoformat()
    return str(value).strip()

def table(records):
    records = iter(records)
    fields = [normalize(v) for v in next(records, ())]
    if not fields or "" in fields or len(set(fields)) != len(fields):
        raise ValueError("يلزم صف عناوين أول، بأسماء أعمدة معبأة ومختلفة.")
    rows = []
    for number, record in enumerate(records, 2):
        if not record or all(normalize(v) == "" for v in record):
            continue
        if len(record) != len(fields):
            raise ValueError(f"عدد القيم في الصف {number} مختلف عن عدد العناوين. تحقّق من فاصل CSV أو عرض الجدول.")
        rows.append(dict(zip(fields, map(normalize, record))))
    return fields, rows

def read_table(name, content, choose_sheet, encoding="تلقائي", delimiter="تلقائي"):
    ext = Path(name).suffix.lower()
    if ext == ".csv":
        if encoding == "تلقائي":
            if content.startswith((b"\xff\xfe", b"\xfe\xff")):
                text = content.decode("utf-16")
            else:
                try:
                    text = content.decode("utf-8-sig")
                except UnicodeDecodeError:
                    text = content.decode("cp1256")
        else:
            text = content.decode(encoding)
        if delimiter == "تلقائي":
            try:
                delimiter = csv.Sniffer().sniff(text[:16384], delimiters=",;\t|").delimiter
            except csv.Error:
                delimiter = ","
        fields, rows = table(csv.reader(io.StringIO(text, newline=""), delimiter=delimiter, strict=True))
        return fields, rows, None
    if ext in (".xlsx", ".xlsm"):
        from openpyxl import load_workbook
        book = load_workbook(io.BytesIO(content), read_only=True, data_only=True)
        try:
            sheet = choose_sheet(book.sheetnames)
            def excel_value(cell):
                value = cell.value
                fmt = getattr(cell, "number_format", "") or ""
                if type(value) is int and re.fullmatch(r"0+", fmt):
                    return str(value).zfill(len(fmt))
                return value
            fields, rows = table([excel_value(c) for c in row] for row in book[sheet].rows)
            return fields, rows, sheet
        finally:
            book.close()
    if ext == ".xls":
        try:
            import xlrd
        except ImportError:
            import subprocess, sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xlrd>=2.0.1,<3"])
            import xlrd
        book = xlrd.open_workbook(file_contents=content)
        try:
            sheet = choose_sheet(book.sheet_names())
            ws = book.sheet_by_name(sheet)
            def legacy(cell):
                if cell.ctype == xlrd.XL_CELL_DATE:
                    return xlrd.xldate_as_datetime(cell.value, book.datemode)
                if cell.ctype == xlrd.XL_CELL_NUMBER and cell.value.is_integer():
                    return int(cell.value)
                return cell.value
            fields, rows = table([legacy(cell) for cell in ws.row(i)] for i in range(ws.nrows))
            return fields, rows, sheet
        finally:
            book.release_resources()
    raise ValueError("اختر ملف CSV أو Excel بامتداد xlsx أو xlsm أو xls.")

def parse_day(value):
    try:
        return datetime.fromisoformat(value).date()
    except ValueError:
        return None

def finite_number(value):
    try:
        n = float(value)
        return n if math.isfinite(n) else None
    except ValueError:
        return None

def suggested(fields):
    aliases = {
        "customer_id":"customer_id", "معرف العميل":"customer_id", "رقم العميل":"customer_id",
        "email":"email", "email_address":"email", "البريد الإلكتروني":"email", "البريد الالكتروني":"email",
        "mobile_number":"mobile_number", "رقم الجوال":"mobile_number", "الجوال":"mobile_number",
        "national_id":"national_id", "رقم الهوية":"national_id", "full_name":"full_name", "الاسم الكامل":"full_name",
        "city":"city", "المدينة":"city", "customer_status":"customer_status", "حالة العميل":"customer_status",
        "date_of_birth":"date_of_birth", "تاريخ الميلاد":"date_of_birth", "registration_date":"registration_date",
        "تاريخ التسجيل":"registration_date", "last_updated_at":"last_updated_at", "آخر تحديث":"last_updated_at",
        "تاريخ آخر تحديث":"last_updated_at", "تاريخ التحديث":"last_updated_at"}
    roles = {f: aliases.get(f.strip().casefold()) for f in fields}
    course = set(aliases.values()).issubset(roles.values())
    configs = {}
    for f, role in roles.items():
        kind = "حر" 
        allowed = ""
        if role == "email":
            kind = "بريد"
        elif role in ("date_of_birth", "registration_date", "last_updated_at"):
            kind = "تاريخ"
        if course:
            kind = {"customer_id":"كود العميل التدريبي", "mobile_number":"جوال سعودي",
                    "national_id":"هوية تدريبية", "full_name":"اسم كامل",
                    "city":"قائمة", "customer_status":"قائمة"}.get(role, kind)
            if role == "city":
                allowed = "الرياض|جدة|الدمام|مكة المكرمة|المدينة المنورة|أبها|تبوك"
            if role == "customer_status":
                allowed = "نشط|غير نشط|معلق"
        configs[f] = dict(required=True, unique=(course and role in ("customer_id","email","mobile_number","national_id")),
                          kind=kind, allowed=allowed, recent=(role=="last_updated_at"),
                          minimum="2022-01-01" if course and role=="registration_date" else "")
    byrole = {r:f for f,r in roles.items() if r}
    relations = []
    if course:
        relations = [(byrole["date_of_birth"], "<", byrole["registration_date"]),
                     (byrole["registration_date"], "≤", byrole["last_updated_at"])]
    return configs, byrole.get("customer_id"), relations

In [ ]:
from __future__ import annotations

import csv
import random
from datetime import date, timedelta
from pathlib import Path


PROJECT_ROOT = Path.cwd() / "customer_quality_run"
DATA_DIR = PROJECT_ROOT / "data"
RAW_FILE = DATA_DIR / "customers_raw.csv"
REFERENCE_FILE = DATA_DIR / "customers_clean_reference.csv"
RANDOM_SEED = 20261118
CUSTOMER_COUNT = 120

FIELDNAMES = [
    "customer_id",
    "full_name",
    "email",
    "mobile_number",
    "national_id",
    "city",
    "date_of_birth",
    "registration_date",
    "last_updated_at",
    "customer_status",
]

FIRST_NAMES = [
    "أحمد", "سارة", "محمد", "نورة", "خالد", "ريم", "عبدالله", "الجوهرة",
    "فهد", "دانة", "تركي", "مها", "عمر", "ليان", "ماجد", "هيا",
]
LAST_NAMES = [
    "العتيبي", "القحطاني", "الغامدي", "الحربي", "المطيري", "الشهري",
    "الزهراني", "الشمري", "الدوسري", "السبيعي",
]
CITIES = ["الرياض", "جدة", "الدمام", "مكة المكرمة", "المدينة المنورة", "أبها", "تبوك"]
STATUSES = ["نشط", "غير نشط", "معلق"]


def iso_date_between(start: date, end: date) -> str:
    return (start + timedelta(days=random.randint(0, (end - start).days))).isoformat()


def build_clean_record(index: int) -> dict[str, str]:
    first_name = random.choice(FIRST_NAMES)
    last_name = random.choice(LAST_NAMES)
    full_name = f"{first_name} {last_name}"
    latin_email = f"customer{index:03d}@example.test"
    birth_date = iso_date_between(date(1965, 1, 1), date(2004, 12, 31))
    registration_date = iso_date_between(date(2022, 1, 1), date(2026, 8, 31))
    return {
        "customer_id": f"CUST-{index:05d}",
        "full_name": full_name,
        "email": latin_email,
        "mobile_number": f"05{random.randint(0, 9)}{random.randint(1000000, 9999999)}",
        "national_id": f"1{random.randint(100000000, 999999999)}",
        "city": random.choice(CITIES),
        "date_of_birth": birth_date,
        "registration_date": registration_date,
        "last_updated_at": iso_date_between(date(2026, 9, 1), date(2026, 11, 18)),
        "customer_status": random.choice(STATUSES),
    }


def introduce_quality_issues(records: list[dict[str, str]]) -> list[dict[str, str]]:
    """Return a copy with deterministic, intentional quality issues."""
    raw = [record.copy() for record in records]

    # Completeness: mandatory fields missing.
    raw[2]["email"] = ""
    raw[7]["mobile_number"] = ""
    raw[15]["city"] = ""
    raw[26]["national_id"] = ""

    # Validity: invalid syntax or values outside the controlled domain.
    raw[5]["email"] = "not-an-email"
    raw[11]["mobile_number"] = "056123"  # Too short.
    raw[18]["national_id"] = "ABC12345"
    raw[33]["city"] = "الرياضض"
    raw[41]["customer_status"] = "قيد المراجعة"
    raw[52]["date_of_birth"] = "1991/13/40"

    # Consistency: values that disagree with another field.
    raw[20]["date_of_birth"] = "2030-05-10"  # Date of birth after registration.
    raw[46]["registration_date"] = "2021-12-31"  # Outside expected business period.

    # Timeliness: source has not been refreshed within the intended 90-day SLA.
    raw[24]["last_updated_at"] = "2025-08-12"
    raw[70]["last_updated_at"] = "2026-01-05"

    # Accuracy: differs from the reference value; requires authoritative source.
    raw[30]["full_name"] = "اسم غير مطابق"
    raw[61]["city"] = "جدة" if records[61]["city"] != "جدة" else "الرياض"

    # Uniqueness: duplicate business identifiers. Retain different record IDs to
    # represent duplicate customer registrations, not merely duplicate rows.
    raw[80]["email"] = raw[79]["email"]
    raw[95]["national_id"] = raw[94]["national_id"]
    raw[110]["mobile_number"] = raw[109]["mobile_number"]

    return raw


def write_csv(path: Path, records: list[dict[str, str]]) -> None:
    with path.open("w", encoding="utf-8-sig", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=FIELDNAMES)
        writer.writeheader()
        writer.writerows(records)


def main() -> None:
    random.seed(RANDOM_SEED)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    clean_records = [build_clean_record(index) for index in range(1, CUSTOMER_COUNT + 1)]
    raw_records = introduce_quality_issues(clean_records)
    write_csv(REFERENCE_FILE, clean_records)
    write_csv(RAW_FILE, raw_records)
    print(f"Created {len(raw_records)} raw customer records: {RAW_FILE}")
    print(f"Created {len(clean_records)} reference records: {REFERENCE_FILE}")

In [ ]:
from statistics import quantiles

KINDS = ["حر", "عدد", "عدد غير سالب", "تاريخ", "بريد", "قائمة", "جوال سعودي", "هوية تدريبية", "اسم كامل", "كود العميل التدريبي"]
PATTERNS = {"بريد": r"[^\s@]+@[^\s@]+\.[^\s@]+", "جوال سعودي": r"05[0-9]{8}",
            "هوية تدريبية": r"[12][0-9]{9}", "كود العميل التدريبي": r"CUST-[0-9]{5}"}
ACTIONS = ["استكمل الحقول المطلوبة من المصدر.", "صحّح الصيغة أو القيمة وفق القاعدة.",
           "راجع طرفي التكرار قبل دمج أي سجل.", "طابق القيمتين المرتبطتين مع المصدر.",
           "تحقق من الاختلاف مع صاحب المرجع.", "حدّث السجل وتحقق من تاريخ التحديث."]

def evaluate(fields, rows, configs, key=None, ref=None, ref_key=None, relations=(), asof=None, days=90):
    asof = asof or datetime.now(ZoneInfo("Asia/Riyadh")).date()
    if days < 0:
        raise ValueError("حد الحداثة يجب أن يكون صفرًا أو أكثر.")
    ref_fields, ref_rows = ref if ref is not None else ([], [])
    if ref is not None and (key not in fields or ref_key not in ref_fields):
        raise ValueError("اختر مفتاح ربط من كل ملف لقياس الدقة.")
    keys = Counter(r[key] for r in rows) if key else {}
    refs = {}
    for r in ref_rows:
        k = r[ref_key]
        refs[k] = None if k in refs else r
    for a, op, b in relations:
        if a not in fields or b not in fields or a == b or op not in ("=", "<", "≤"):
            raise ValueError("تحقق من الحقول والعامل في قاعدة الاتساق.")
    cells, signals = [], []
    def add(f, dimension, description, check=None, status="لا ينطبق"):
        good = checked = 0
        issues = []
        if check is not None:
            for i, row in enumerate(rows, 1):
                answer = check(row)
                if answer is None:
                    continue
                checked += 1
                if answer:
                    good += 1
                else:
                    issues.append({"row": i})
        cells.append(dict(field=f, dimension=DIMS[dimension], rule=description, passed=good, evaluated=checked,
                          excluded=len(rows)-checked, issues=issues, action=ACTIONS[dimension],
                          percent=round(100*good/checked, 2) if checked else None,
                          status="تم القياس" if checked else ("غير قابل للقياس" if check else status)))
    for f in fields:
        cfg = configs[f]
        kind = cfg["kind"]
        add(f, 0, "قيمة مطلوبة بعد إزالة المسافات الطرفية" if cfg["required"] else "الحقل اختياري",
            (lambda r: bool(r[f])) if cfg["required"] else None)
        options = {v.strip() for v in cfg["allowed"].split("|") if v.strip()}
        minimum = cfg.get("minimum","").strip()
        if minimum and (kind!="تاريخ" or not parse_day(minimum)):
            raise ValueError(f"راجع الحد الأدنى للتاريخ في الحقل {f}.")
        if kind=="قائمة" and not options:
            raise ValueError(f"حدد القيم المسموحة في {f} وافصل بينها بعلامة |")
        def valid(r):
            v = r[f]
            if not v:
                return None
            if kind in PATTERNS:
                return bool(re.fullmatch(PATTERNS[kind], v))
            if kind=="قائمة":
                return v in options
            if kind=="تاريخ":
                day = parse_day(v)
                return day is not None and (not minimum or day >= parse_day(minimum))
            if kind=="اسم كامل":
                return len(v.split()) >= 2 and all(ch.isalpha() or ch.isspace() for ch in v)
            n = finite_number(v)
            return n is not None and (kind!="عدد غير سالب" or n>=0)
        desc = {"حر":"لم تُحدَّد قاعدة صيغة لهذا الحقل", "تاريخ":"تاريخ ISO أو خلية تاريخ Excel",
                "قائمة":"ضمن القيم: "+cfg["allowed"]}.get(kind,kind)
        add(f, 1, desc + (" ابتداءً من "+minimum if minimum else ""), valid if kind!="حر" else None)
        counts = Counter(r[f] for r in rows if r[f]) if cfg["unique"] else {}
        add(f, 2, "تُعلَّم جميع حالات القيمة المكررة" if cfg["unique"] else "التكرار مسموح وفق الإعداد",
            (lambda r: counts[r[f]]==1 if r[f] else None) if cfg["unique"] else None)
        related = [(a,op,b) for a,op,b in relations if f in (a,b)]
        def consistent(r):
            decisions = []
            for a,op,b in related:
                left, right = r[a], r[b]
                if not left or not right:
                    decisions.append(None)
                    continue
                if op=="=":
                    decisions.append(left==right)
                else:
                    x,y = parse_day(left),parse_day(right)
                    decisions.append(None if x is None or y is None else (x<y if op=="<" else x<=y))
            return False if False in decisions else (None if None in decisions else True)
        add(f, 3, " ؛ ".join(f"{a} {op} {b}" for a,op,b in related) or "لم تُضبط علاقة لهذا الحقل", consistent if related else None)
        def accurate(r):
            k = r[key]
            other = refs.get(k)
            if not k or not other or keys[k]!=1 or not r[f] or not other[f]:
                return None
            return r[f]==other[f]
        accuracy_ok = ref is not None and f!=key and f in ref_fields
        reason = "مطابقة مع المرجع بعد الربط بمفتاح فريد في المصدرين" if accuracy_ok else (
            "أرفق ملفًا مرجعيًا لقياس الدقة" if ref is None else "مفتاح ربط أو حقل لا مقابل له في المرجع")
        add(f, 4, reason, accurate if accuracy_ok else None, "غير قابل للقياس")
        def recent(r):
            dt = parse_day(r[f])
            return 0 <= (asof-dt).days <= days if dt else None
        add(f, 5, f"عمر التحديث بين 0 و{days} يومًا عند {asof}" if cfg["recent"] else "لم يُعيَّن كتاريخ تحديث",
            recent if cfg["recent"] else None)
        if kind in ("عدد","عدد غير سالب"):
            nums = [(i,n) for i,r in enumerate(rows,1) if (n:=finite_number(r[f])) is not None]
            if len(nums)>=8:
                q1,_,q3 = quantiles([n for _,n in nums],n=4,method="inclusive")
                margin = 1.5*(q3-q1)
                signals.extend({"field":f,"row":i} for i,n in nums if n<q1-margin or n>q3+margin)
    return dict(fields=fields, dimensions=DIMS, cells=cells, rows=len(rows), snapshot=asof.isoformat(), signals=signals)

In [ ]:
"""Render the assessment JSON as a self-contained Arabic HTML dashboard."""
from html import escape
from pathlib import Path
import json
import sys


PAGE = """<!doctype html><html lang="ar" dir="rtl"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>لوحة تقرير جودة البيانات</title>
<style>
*{box-sizing:border-box}body{margin:0;background:#f5f6f2;color:#183a38;font:16px/1.65 Tahoma,Arial,sans-serif}
main{max-width:1280px;margin:auto;padding:28px clamp(12px,4vw,54px)}header{border-bottom:3px solid #1b8177;padding-bottom:18px}
h1,h2,p{margin:.25em 0}h1{font-size:clamp(24px,4vw,40px)}h2{font-size:21px}section{margin-top:24px}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));gap:12px}.card,.panel{background:white;border:1px solid #d1dcd5;border-radius:12px;padding:16px}
.card strong{display:block;font:700 30px/1.4 Arial;color:#12665f}.scroll{overflow:auto}table{width:100%;border-collapse:collapse;white-space:nowrap}
td,th{padding:8px;text-align:center;border-bottom:1px solid #e3e9e5}th[scope=row]{text-align:start}
thead{background:#eff4ef}button,select{font:inherit}td button{border:1px solid transparent;border-radius:6px;padding:5px 8px;width:100%;cursor:pointer;color:#174d46;background:#dff1e6}
td button.problem{background:#fff0d3;color:#734900}td button.na{background:#f1f2f3;color:#505e60;font-size:13px}
td button:focus-visible,td button.active{outline:2px solid #185d73;outline-offset:1px}label{display:inline-block;margin:8px 0 8px 18px}select{padding:4px}
code,bdi{direction:ltr;unicode-bidi:isolate}#issues{max-height:200px;overflow:auto}#issues li{display:inline-block;margin:3px;padding:4px 8px;background:#fff3de;border-radius:4px}#issues{padding:0}
#queue button{margin:3px;padding:7px 12px;border:1px solid #a9bbb5;border-radius:7px;color:#184844;background:#f0f6f2;cursor:pointer}#outliers{overflow-wrap:anywhere}</style><main><header><h1>تقرير جودة البيانات</h1><p><span id="source"></span> · تاريخ القياس <bdi id="day"></bdi></p></header>
<section class="kpis" aria-label="ملخص"><div class="card">عدد السجلات<strong id="rows"></strong></div><div class="card">القواعد التي رُصدت لها مخالفات<strong id="broken"></strong></div><div class="card">اكتمال الخانات الإلزامية<strong id="complete"></strong></div><div class="card">سجلات لها مخالفة واحدة على الأقل<strong id="affected"></strong></div></section>
<section class="panel"><h2>أولويات المعالجة</h2><p>القواعد مرتبة بعدد السجلات المخالفة. اختر قاعدة لعرض تفاصيلها.</p><div id="queue"></div></section><section class="panel"><h2>مصفوفة جودة الحقول</h2><p>اختر نسبة لعرض القاعدة والسجلات المخالفة. النسبة = عدد القيم المطابقة ÷ عدد القيم المفحوصة.</p>
<div class="scroll"><table><thead id="head"></thead><tbody id="matrix"></tbody></table></div></section>
<section id="detail" class="panel" tabindex="-1"><h2>تفاصيل القياس</h2><label>الحقل <select id="field" dir="ltr"></select></label><label>البُعد <select id="dim"></select></label>
<div aria-live="polite"><p id="rule"></p><p id="stat"></p><ul id="issues"></ul><p id="action"></p></div></section><section class="panel"><h2>قيم عددية تستحق المراجعة</h2><p>قيم خارج حدود 1.5 × المدى الربيعي، عند توفر ثماني قيم عددية على الأقل. تُعرض منفصلة عن مخالفات القواعد.</p><div id="outliers"></div></section></main>
<script type="application/json" id="payload">__REPORT__</script><script>
const data=JSON.parse(document.getElementById('payload').textContent);
const el=id=>document.getElementById(id);
function add(tag,parent,text){const node=document.createElement(tag);parent.appendChild(node);if(text!==undefined)node.textContent=text;return node}
const complete=data.cells.filter(c=>c.dimension===data.dimensions[0]);
const total=complete.reduce((n,c)=>n+c.evaluated,0), filled=complete.reduce((n,c)=>n+c.passed,0);
el('rows').textContent=data.rows;el('day').textContent=data.snapshot;el('complete').textContent=total?(100*filled/total).toFixed(2)+'%':'غير قابل للقياس';
el('affected').textContent=new Set(data.cells.flatMap(c=>c.issues.map(i=>i.row))).size;
for(const f of data.fields) el('field').add(new Option(f,f));
for(const d of data.dimensions) el('dim').add(new Option(d,d));
const header=add('tr',el('head'));for(const name of ['الحقل',...data.dimensions])add('th',header,name).scope='col';
function show(){
 const f=el('field').value,d=el('dim').value,c=data.cells.find(x=>x.field===f&&x.dimension===d);
 el('rule').textContent=c.rule;
 el('stat').textContent=c.percent===null?c.status+' · عدد السجلات المفحوصة 0':'المطابق '+c.passed+' من '+c.evaluated+' مفحوصًا · المخالف '+c.issues.length+' · المستبعد من المقام '+c.excluded;
 el('issues').replaceChildren();
 el('action').textContent=c.issues.length?c.action+' · يُعرض أول 50 رقم سجل، والقائمة الكاملة في JSON.':'';
 for(const i of c.issues.slice(0,50))add('li',el('issues'),'السجل '+i.row);
 for(const b of el('matrix').querySelectorAll('button'))b.classList.toggle('active',b.dataset.f===f&&b.dataset.d===d);
}
for(const f of data.fields){
 const tr=add('tr',el('matrix'));const title=add('th',tr,f);title.scope='row';title.dir='ltr';
 for(const d of data.dimensions){
 const c=data.cells.find(x=>x.field===f&&x.dimension===d),td=add('td',tr),label=c.percent===null?c.status:c.percent.toFixed(2)+'%';
 const b=add('button',td,label);b.dataset.f=f;b.dataset.d=d;b.className=c.percent===null?'na':c.issues.length?'problem':'';
 b.setAttribute('aria-label',f+'، '+d+'، '+label);
 b.onclick=()=>{el('field').value=f;el('dim').value=d;show();el('detail').focus()};}}
el('field').onchange=el('dim').onchange=show;el('field').value=data.fields[0];el('dim').value=data.dimensions[0];show();
el('source').textContent=data.source; const flagged=data.cells.filter(c=>c.issues.length).sort((a,b)=>b.issues.length-a.issues.length);
el('broken').textContent=flagged.length;
if(!flagged.length)el('queue').textContent='لم تُرصد مخالفات في القواعد التي أمكن قياسها.';
for(const c of flagged){const b=add('button',el('queue'),c.field+' · '+c.dimension+' · '+c.issues.length+' مخالف');b.onclick=()=>{el('field').value=c.field;el('dim').value=c.dimension;show();el('detail').focus()}}
el('outliers').textContent=data.signals.length?data.signals.length+' حالة. '+data.signals.slice(0,50).map(s=>s.field+' (السجل '+s.row+')').join('، '):'لا توجد إشارات ضمن الحقول العددية المؤهلة.';
</script></html>"""


def render_dashboard(report_path, output_path):
    report = json.loads(Path(report_path).read_text(encoding="utf-8"))
    payload = json.dumps(report, ensure_ascii=False).replace("<", "\\u003c")
    Path(output_path).write_text(PAGE.replace("__REPORT__", payload), encoding="utf-8")

In [ ]:
#@title مصدر البيانات
MODE = "رفع ملف" #@param ["رفع ملف", "مثال العملاء"]
WITH_REFERENCE = False #@param {type:"boolean"}
CSV_ENCODING = "تلقائي" #@param ["تلقائي","utf-8-sig","cp1256","utf-16","cp1252"]
CSV_SEPARATOR = "تلقائي" #@param ["تلقائي",",",";","Tab","|"]
from google.colab import files
from pathlib import Path

def pick_sheet(names):
    if len(names)==1:
        return names[0]
    for index,name in enumerate(names,1):
        print(index,name)
    while True:
        choice = input("أدخل رقم ورقة العمل: ").strip()
        if choice.isdecimal() and 1<=int(choice)<=len(names):
            return names[int(choice)-1]

def pick_file(message):
    print(message)
    selected=files.upload()
    if len(selected)!=1:
        raise ValueError("اختر ملفًا واحدًا في هذه الخطوة، ثم أعد تشغيل خلية مصدر البيانات.")
    name,content=next(iter(selected.items()))
    columns,records,sheet=read_table(name,content,pick_sheet,CSV_ENCODING,"\t" if CSV_SEPARATOR=="Tab" else CSV_SEPARATOR)
    label=Path(name).name + (" · "+sheet if sheet else "")
    print(label,"| عدد الأعمدة",len(columns),"| عدد السجلات",len(records))
    return label,columns,records

reference=None
reference_label=None
if MODE=="رفع ملف":
    source_label,fields,rows=pick_file("اختر ملف البيانات الأساسي")
    if WITH_REFERENCE:
        reference_label,ref_columns,ref_rows=pick_file("اختر الملف المرجعي")
        reference=(ref_columns,ref_rows)
else:
    main()
    source_label,reference_label="مثال العملاء","مرجع العملاء التدريبي"
    fields,rows,_=read_table(RAW_FILE.name,RAW_FILE.read_bytes(),pick_sheet)
    ref_columns,ref_rows,_=read_table(REFERENCE_FILE.name,REFERENCE_FILE.read_bytes(),pick_sheet)
    reference=(ref_columns,ref_rows)

In [ ]:
# اختبارات مرجع الدورة تخص وضع مثال العملاء
if MODE == "مثال العملاء":
    
    import csv
    import re
    import unittest
    from datetime import date
    from pathlib import Path
    
    
    RAW_PATH = DATA_DIR / "customers_raw.csv"
    REFERENCE_PATH = DATA_DIR / "customers_clean_reference.csv"
    EXPECTED_FIELDS = [
        "customer_id", "full_name", "email", "mobile_number", "national_id",
        "city", "date_of_birth", "registration_date", "last_updated_at", "customer_status",
    ]
    VALID_CITIES = {"الرياض", "جدة", "الدمام", "مكة المكرمة", "المدينة المنورة", "أبها", "تبوك"}
    VALID_STATUSES = {"نشط", "غير نشط", "معلق"}
    
    
    def read_csv(path: Path) -> list[dict[str, str]]:
        with path.open(encoding="utf-8-sig", newline="") as csv_file:
            return list(csv.DictReader(csv_file))
    
    
    class GeneratedCustomerDataTests(unittest.TestCase):
        @classmethod
        def setUpClass(cls) -> None:
            cls.reference = read_csv(REFERENCE_PATH)
            cls.raw = read_csv(RAW_PATH)
    
        def test_both_files_have_expected_schema_and_size(self) -> None:
            self.assertEqual(len(self.reference), 120)
            self.assertEqual(len(self.raw), 120)
            self.assertEqual(list(self.reference[0]), EXPECTED_FIELDS)
            self.assertEqual(list(self.raw[0]), EXPECTED_FIELDS)
    
        def test_reference_data_is_valid_baseline(self) -> None:
            for record in self.reference:
                self.assertRegex(record["customer_id"], r"^CUST-\d{5}$")
                self.assertRegex(record["email"], r"^customer\d{3}@example\.test$")
                self.assertRegex(record["mobile_number"], r"^05\d{8}$")
                self.assertRegex(record["national_id"], r"^1\d{9}$")
                self.assertIn(record["city"], VALID_CITIES)
                self.assertIn(record["customer_status"], VALID_STATUSES)
                self.assertLess(date.fromisoformat(record["date_of_birth"]), date.fromisoformat(record["registration_date"]))
    
        def test_raw_data_keeps_record_identifiers_for_reference_comparison(self) -> None:
            self.assertEqual(
                [record["customer_id"] for record in self.raw],
                [record["customer_id"] for record in self.reference],
            )
    
        def test_raw_data_contains_known_examples_of_each_dimension(self) -> None:
            self.assertEqual(self.raw[2]["email"], "")
            self.assertEqual(self.raw[5]["email"], "not-an-email")
            self.assertGreater(date.fromisoformat(self.raw[20]["date_of_birth"]), date.fromisoformat(self.raw[20]["registration_date"]))
            self.assertEqual(self.raw[24]["last_updated_at"], "2025-08-12")
            self.assertNotEqual(self.raw[30]["full_name"], self.reference[30]["full_name"])
            self.assertEqual(self.raw[80]["email"], self.raw[79]["email"])
    
    suite = unittest.defaultTestLoader.loadTestsFromTestCase(GeneratedCustomerDataTests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if result.testsRun != 4 or not result.wasSuccessful():
        raise AssertionError("تحقق من الاختبارات الأربعة قبل الانتقال إلى القياس")
else:
    print(f"جاهز لضبط القواعد على {len(fields)} عمودًا في ملفك.")

In [ ]:
import ipywidgets as W
import html
import tempfile
from zipfile import ZipFile
from IPython.display import HTML, display, clear_output
from google.colab import files, output
output.enable_custom_widget_manager()

def launch(source, fields, rows, reference, demo):
    defaults, suggested_key, initial_pairs = suggested(fields)
    layout = W.Layout(width="175px")
    def dropdown(options, value=None):
        return W.Dropdown(options=options, value=value, layout=layout)
    form = {}
    items = []
    for f in fields:
        c = defaults[f]
        required = W.Checkbox(description="مطلوب", value=c["required"], indent=False, layout=W.Layout(width="84px"))
        unique = W.Checkbox(description="فريد", value=c["unique"], indent=False, layout=W.Layout(width="74px"))
        recent = W.Checkbox(description="تحديث", value=c["recent"], indent=False, layout=W.Layout(width="84px"))
        kind = dropdown(KINDS,c["kind"])
        allowed = W.Text(value=c["allowed"], placeholder="القيم المسموحة مفصولة بـ |", layout=layout)
        minimum = W.Text(value=c["minimum"],placeholder="أقدم تاريخ YYYY-MM-DD",layout=layout)
        form[f]=dict(required=required,unique=unique,recent=recent,kind=kind,allowed=allowed,minimum=minimum)
        label=W.HTML("<b>"+html.escape(f)+"</b>",layout=W.Layout(width="150px"))
        items.append(W.Box([label,kind,required,unique,recent,allowed,minimum],layout=W.Layout(flex_flow="row wrap")))
    keys = [("—",None)]+[(f,f) for f in fields]
    key=dropdown(keys,suggested_key)
    ref_fields = reference[0] if reference else []
    ref_key=dropdown([("—",None)]+[(f,f) for f in ref_fields], suggested_key if suggested_key in ref_fields else None)
    day=W.DatePicker(value=date(2026,11,18) if demo else datetime.now(ZoneInfo("Asia/Riyadh")).date(),description="تاريخ القياس")
    days=W.BoundedIntText(value=90,min=0,max=36500,description="أيام الحداثة")
    pairs = []
    relation_list=W.VBox()
    def new_pair(a=None, op="=", b=None):
        x=dropdown(keys,a); cond=dropdown(["=","<","≤"],op); y=dropdown(keys,b)
        delete=W.Button(description="إزالة",layout=W.Layout(width="75px"))
        item=(x,cond,y)
        row=W.HBox([x,cond,y,delete])
        def remove(_):
            pairs.remove(item)
            relation_list.children=tuple(w for w in relation_list.children if w is not row)
        delete.on_click(remove)
        pairs.append(item)
        relation_list.children=(*relation_list.children,row)
    for a,op,b in initial_pairs:
        new_pair(a,op,b)
    add_button=W.Button(description="إضافة علاقة اتساق")
    add_button.on_click(lambda _:new_pair())
    run=W.Button(description="تشغيل الفحص",button_style="success")
    download=W.Button(description="تنزيل التقرير",disabled=True)
    result=W.Output()
    export={"path":None}
    def execute(_):
        with result:
            clear_output(wait=True)
            download.disabled=True
            export["path"]=None
            try:
                config={f:{k:w.value for k,w in control.items()} for f,control in form.items()}
                edges=[(a.value,op.value,b.value) for a,op,b in pairs]
                if day.value is None:
                    raise ValueError("اختر تاريخ القياس.")
                report=evaluate(fields,rows,config,key.value,reference,ref_key.value,edges,day.value,days.value)
                report.update(source=source, rules=config, key=key.value, reference_key=ref_key.value,
                              reference_source=reference_label, relations=edges, max_age_days=days.value)
                folder=Path(tempfile.mkdtemp(prefix="quality_report_"))
                raw=folder/"quality_results.json"
                raw.write_text(json.dumps(report,ensure_ascii=False), encoding="utf-8")
                page=folder/"dashboard.html"
                render_dashboard(raw,page)
                bundle=folder/"quality_report.zip"
                with ZipFile(bundle,"w",compression=8) as pack:
                    for file in (raw,page):
                        pack.write(file,file.name)
                show_html=html.escape(page.read_text(encoding="utf-8"),quote=True)
                display(HTML(f'<iframe title="تقرير جودة البيانات" sandbox="allow-scripts" style="width:100%;height:920px;border:0" srcdoc="{show_html}"></iframe>'))
                print("اكتمل الفحص. استخدم «تنزيل التقرير» لحفظ اللوحة وملف النتائج.")
                export["path"]=bundle
                download.disabled=False
            except (ValueError, OSError) as err:
                print("تعذّر إنشاء التقرير:",err)
    run.on_click(execute)
    download.on_click(lambda _:files.download(str(export["path"])) if export["path"] else None)
    display(W.HTML("<h3>ضبط قواعد الملف</h3><p>راجع الإعدادات المقترحة. اختر «عدد» لفحص الأرقام والإشارات الشاذة أو «قائمة» وحدد القيم المسموحة. فحص الحداثة يتطلب قيم تاريخ قابلة للقراءة.</p>"))
    display(W.VBox(items),day,days)
    if reference is not None:
        display(W.HTML("مفتاح الربط في الملف الأساسي ثم المرجعي. تقارَن الأعمدة المشتركة بالاسم."),W.HBox([key,ref_key]))
    display(W.HTML("علاقات الاتساق: المساواة نصية، وعاملا الترتيب للمقارنة بين تاريخين. تُقرأ العلاقة من الحقل الأول إلى الثاني."),relation_list,add_button,W.HBox([run,download]),result)

launch(source_label, fields, rows, reference, MODE=="مثال العملاء")